In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
import numpy as np
import numpy.typing as npt


from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qopt_best_practices.sat_mapping import SATMapper


from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples

In [2]:
service = QiskitRuntimeService(name='eu_test_instance')
backend = service.least_busy(min_num_qubits=1, operational=True, simulator=False) 

In [3]:
backend

<IBMBackend('ibm_brussels')>

In [4]:
filename = 'test_N2_W2'
N = 2
p = 1
delta_b = 0.1
delta_g = 4.0


data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'


Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

   
swap_strat = SwapStrategy.from_line(list(range(num_qubits)))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(num_qubits)}

qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = 1,
    flatten=True
)
graph = circuit_to_graph(qc, qc.parameters[1])

edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat
)
if remapped_g is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

betas = [(1-k/p) * delta_b for k in range(p)]
gammas = [(k+1) / p * delta_g for k in range(p)]
fixed_params = betas + gammas


phis = ParameterVector('ϕ', num_qubits)
betas = ParameterVector('β', p)


init = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init.ry(phis[i], i)
    
mixer = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    mixer.ry(-phis[i], i)
    mixer.rz(-2*betas[0], i)
    mixer.ry(phis[i], i)
    
circ_dict = circuit_construction(singles, doubles, backend, swap_strat, edge_coloring, {}, p, init, mixer)
# circuit = circ_dict['circuit_to_sample']
circuit = circ_dict['backend']


In [5]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

In [6]:
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")


In [7]:
catalog.list()


[QiskitFunction(qedma/qesem)]

In [8]:
obsvs = [
    SparsePauliOp.from_sparse_list(
        [('Z', [q], 1)], num_qubits=circuit.num_qubits
    ) for q in range(circ_dict['circuit_to_sample'].num_qubits)
][:1]

In [9]:

fixed_param_bind = {circuit.parameters[i]: fixed_params[i] for i in range(2*p)}
fixed_qc = circuit.assign_parameters(fixed_param_bind)

prob = 1 / (2 * N)
theta = 2 * np.arcsin(np.sqrt(prob))
init_angles = theta * np.ones((num_qubits,))
sample_circuit = fixed_qc.assign_parameters(init_angles, inplace=False)

In [10]:
sample_circuit.draw(fold=-1)

global phase: π/4
                ┌────┐      ┌──────────┐   ┌────┐   ┌─────────┐   ┌────┐    ┌────────┐          ┌──────┐  ┌─────────┐   ┌────────┐     ┌────┐     ┌────────┐           ┌──────┐┌───────────────┐┌──────────┐     ┌──────┐       ┌──────────┐     ┌────┐                    ┌──────┐     ┌─────────┐    ┌────┐   ┌──────────┐     ┌──────┐     ┌───────────────┐┌─────────┐             ┌──────┐┌───────────────┐    ┌────────┐   ┌──────────┐           ┌──────┐┌──────────┐   ┌────┐   ┌─────────┐             ┌──────┐┌────────┐      ┌────┐     ┌─────────┐      ┌──────┐       ┌──────────┐        ┌────┐                  ┌──────┐  ┌─────────┐    ┌────┐   ┌──────────┐     ┌──────┐     ┌───────────────┐    ┌─────────┐                ┌──────┐┌───────────────┐    ┌────────┐   ┌──────────┐           ┌──────┐┌──────────┐   ┌────┐   ┌─────────┐             ┌──────┐┌────────┐      ┌────┐     ┌─────────┐     ┌──────┐        ┌──────────┐     ┌────┐                      ┌──────┐     ┌─────────┐      ┌────┐   ┌──────────┐     ┌──────┐     ┌───────────────┐┌─────────┐                  ┌──────┐┌─────────┐  ┌────────┐    ┌────┐    ┌────────┐              ┌──────┐    ┌────┐   ┌──────────┐   ┌────┐    ┌────────┐ ┌──────────┐   ┌────┐   ┌──────────┐  ┌────┐   ┌────────┐                                               ┌─┐                   
 q_2 -> 0 ──────┤ √X ├──────┤ Rz(4π/3) ├───┤ √X ├───┤ Rz(-78) ├───┤ √X ├────┤ Rz(-π) ├──────────┤1     ├──┤ Rz(-10) ├───┤ Rz(-π) ├─────┤ √X ├─────┤ Rz(-π) ├───────────┤1     ├┤ Delay(15[dt]) ├┤ Rz(-π/2) ├─────┤0     ├───────┤ Rz(-π/2) ├─────┤ √X ├────────────────────┤0     ├─────┤ Rz(π/2) ├────┤ √X ├───┤ Rz(-π/2) ├─────┤0     ├─────┤ Delay(15[dt]) ├┤ Rz(π/2) ├─────────────┤1     ├┤ Delay(15[dt]) ├────┤ Rz(40) ├───┤ Rz(-π/2) ├───────────┤1     ├┤ Rz(-π/2) ├───┤ √X ├───┤ Rz(π/2) ├─────────────┤1     ├┤ Rz(-π) ├──────┤ √X ├─────┤ Rz(π/2) ├──────┤0     ├───────┤ Rz(-π/2) ├────────┤ √X ├──────────────────┤0     ├──┤ Rz(π/2) ├────┤ √X ├───┤ Rz(-π/2) ├─────┤0     ├─────┤ Delay(15[dt]) ├────┤ Rz(π/2) ├────────────────┤1     ├┤ Delay(15[dt]) ├────┤ Rz(40) ├───┤ Rz(-π/2) ├───────────┤1     ├┤ Rz(-π/2) ├───┤ √X ├───┤ Rz(π/2) ├─────────────┤1     ├┤ Rz(-π) ├──────┤ √X ├─────┤ Rz(π/2) ├─────┤0     ├────────┤ Rz(-π/2) ├─────┤ √X ├──────────────────────┤0     ├─────┤ Rz(π/2) ├──────┤ √X ├───┤ Rz(-π/2) ├─────┤0     ├─────┤ Delay(15[dt]) ├┤ Rz(π/2) ├──────────────────┤1     ├┤ Rz(-10) ├──┤ Rz(-π) ├────┤ √X ├────┤ Rz(-π) ├──────────────┤1     ├────┤ √X ├───┤ Rz(2π/3) ├───┤ √X ├────┤ Rz(3π) ├─┤ Rz(-0.2) ├───┤ √X ├───┤ Rz(4π/3) ├──┤ √X ├───┤ Rz(3π) ├───────────────────────────────────────────────┤M├───────────────────
          ┌─────┴────┴────┐ └──┬────┬──┘┌──┴────┴──┐└──┬────┬─┘┌──┴────┴──┐┌┴────────┤ ┌──────┐ │      │  └──┬───┬──┘  ┌┴────────┴┐    └────┘     └────────┘  ┌──────┐ │      │└─────┬────┬────┘└──────────┘     │  Ecr │    ┌──┴──────────┴─┐ ┌─┴────┴┐  ┌─────────┐      │  Ecr │     ├─────────┴┐   ├────┤   ├─────────┬┘     │  Ecr │     └───┬────────┬──┘└──┬────┬─┘ ┌──────────┐│      │└────┬──────┬───┘   ┌┴────────┴┐  └──┬────┬──┘           │      │└─┬──────┬─┘┌──┴────┴─┐ └──┬────┬─┘ ┌──────────┐│      │└┬──────┬┘┌─────┴────┴────┐├─────────┤      │  Ecr │    ┌──┴──────────┴─┐    ┌─┴────┴─┐   ┌─────────┐  │  Ecr │  ├─────────┴┐   ├────┤   ├─────────┬┘     │  Ecr │     └───┬────────┬──┘    └──┬────┬─┘    ┌──────────┐│      │└────┬──────┬───┘   ┌┴────────┴┐  └──┬────┬──┘           │      │└─┬──────┬─┘┌──┴────┴─┐ └──┬────┬─┘ ┌──────────┐│      │└┬──────┬┘┌─────┴────┴────┐├─────────┤     │  Ecr │     ┌──┴──────────┴─┐ ┌─┴────┴┐     ┌─────────┐     │  Ecr │     ├─────────┴┐     ├────┤   ├─────────┬┘     │  Ecr │     └───┬────────┬──┘└──┬────┬─┘   ┌──────────┐   │      │└─────────┘  └┬──────┬┘    ├───┬┘   ┌┴────────┴┐             │      │    └────┘   └─┬──────┬─┘ ┌─┴────┴─┐  └─┬────┬─┘ └┬────────┬┘┌──┴────┴──┐└──┬────┬──┘┌─┴────┴─┐┌┴────────┴┐   ┌────┐   ┌──────────┐   ┌────┐   ┌────────┐└╥┘                ┌─┐
 q_3 -> 1 ┤ Delay(15[d

In [11]:
qesem_function = catalog.load("qedma/qesem")
if qesem_function is None:
    raise Exception()


time_estimation_job = qesem_function.run(
    pubs=[(sample_circuit, obsvs)],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend.name,
)
 

In [12]:
time_estimation_job.result()

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 34200, 'description': None, 'total_qpu_time': 0})], metadata={})

In [13]:
16200 // 60

270

In [14]:
sample_circuit.count_ops()

OrderedDict([('rz', 270),
             ('delay', 155),
             ('sx', 130),
             ('ecr', 61),
             ('x', 15),
             ('measure', 8)])

In [15]:
# results = time_estimation_job.result()[0]
# print(f"Mitigated expectation values: \n{results.data.evs}")
# print(f"Mitigated error-bar: \n{results.data.stds}")
# noisy_results = results.metadata["noisy_results"]
# print(f"Noisy expectation values: \n{noisy_results.evs}")
# print(f"Noisy error-bar: \n{noisy_results.stds}")
# print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
# print(
#     f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
# )
# print(
#     f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
# )
# print("Transpiled circuits:")
# for i, circuit in enumerate(results.metadata["transpiled_circs"]):
#     print(f"Circuit {i}:")
#     print(f"  Circuit: \n {circuit['circuit']}")
#     print(f"  Qubit mapping: \n {circuit['qubit_map']}")
#     print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")